[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# Shapes and Broadcasting


## What you will be able to do

Change a tensor's shape without changing its values, reduce it along one dimension, and say what
shape broadcasting will give before you run a sum, in PyTorch and in TensorFlow.

## The idea

### The problem

The stations' features are a table of 1,200 rows and 4 columns, and a model wants exactly that
shape, with a first dimension for the stations and a second for their features. Many mistakes in a
network's code are mistakes of shape. Pictures often arrive with their color channels last, where
PyTorch's layers want them second. A mean taken over the wrong dimension gives 1,200 numbers instead
of 4.

The worst shape mistakes raise nothing. Subtract a column of 1,200 predictions from a vector of
1,200 answers and both frameworks return a table of 1,200 by 1,200 differences, 1.44 million numbers
where 1,200 were meant, and the mean of that table is a number like any other.

### A formal definition

> **Broadcasting** is the rule by which an operation on two tensors of different shapes treats them
> as one shape. The shapes are compared from their last dimension backward. Two sizes fit when they
> are equal or when one of them is 1, and a dimension that one tensor lacks counts as 1. A dimension
> of size 1 is stretched to the other tensor's size, without copying any values.

### Why it works that way

A tensor's values sit in memory in one long row, last dimension fastest, and its shape says how to
group them. Reshaping changes the grouping and keeps the order, so the sizes must multiply to the
same total, and a size of `-1` asks the framework to work one out. PyTorch's `view` reshapes without
copying, which it can do only when the values already sit in the order the new shape needs. In
PyTorch, a transpose swaps dimensions by changing how the memory is read rather than moving any
values, so a transposed tensor can need a copy before it takes a new shape, and `reshape` makes one
when it must. TensorFlow's `tf.transpose` writes a new tensor with the values moved.

Layers treat the first dimension as the batch, one entry per example. A Keras model rejects a
single example that lacks it, while PyTorch's `nn.Linear` accepts one, so adding a dimension of size
1 in front is the form that works in both. Reductions such as `mean` and `max` name the dimension that
disappears, `dim` in PyTorch and `axis` in TensorFlow and NumPy, and `keepdim` or `keepdims` keeps
it at size 1 so that the result broadcasts back against the tensor it came from.

Broadcasting follows NumPy's rules in both frameworks. It makes standardizing a table one line, since a row of 4 means stretches over 1,200 rows, and the same rule turns a column
minus a vector into a square table.

### Where this shows up

The **Reading Shape Errors** notebook reads what layers print when a shape is wrong, the
**Convolutional Networks** notebook moves the channels of an image between second and last place,
and the **Recurrent Networks** notebook gives a window of hourly readings a last dimension of size 1
for its one feature.

### What this notebook covers

- reshaping, and PyTorch's `view`
- a dimension of size 1 added or removed, for a batch of one
- dimensions swapped with `permute` and `transpose`
- reductions over one dimension, with the dimension kept or dropped
- broadcasting, and the stations standardized with it
- tensors joined with `cat`, `concat` and `stack`

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import torch

table = torch.ones(1200, 4)
print(table.mean(dim=0).shape, table.mean(dim=1).shape)
print((table - table.mean(dim=0)).shape)
print((torch.ones(1200, 1) - torch.ones(1200)).shape)
```

```
torch.Size([4]) torch.Size([1200])
torch.Size([1200, 4])
torch.Size([1200, 1200])
```

A mean over dimension 0 leaves one number per column, and a mean over dimension 1 one number per
row. The row of 4 column means broadcasts over all 1,200 rows. The last line raises nothing and
returns 1,440,000 values.

## Setup

Three libraries, a way to quiet TensorFlow while it loads, and the stations.

- `numpy` builds the stations, and its `std` is compared with the frameworks' in one example
- `torch` is PyTorch
- `tensorflow` is TensorFlow
- `contextlib` and `os` hide the messages TensorFlow prints while it loads

TensorFlow's start-up messages carry the time and a process number, so Setup hides them with
`TF_CPP_MIN_LOG_LEVEL` and `c_messages_hidden`, as in the **Why PyTorch and TensorFlow** notebook.

`station_network_arrays` makes the same made-up weather stations as that notebook, drawn from a
fixed seed so they are the same on every run. `features` has 1,200 rows, one per station, with its
latitude, longitude, elevation in meters and distance from the coast in kilometers, and `jan_mean`
holds each station's January mean in degrees Celsius, both as 32-bit floats. The cell also splits
the stations and standardizes their features into `scaled`, which two of the Common errors use.

In [1]:
import contextlib
import os

import numpy as np
import torch


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf


def station_network_arrays(n=1200, seed=2025):
    """The made-up stations of the scikit-learn guide, as arrays: four features and two targets."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    shift = np.where(coast_km < 3, 0.8,
            np.where(elevation > 700, -0.5,
            np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), 0.3, 0.0)))
    rng.choice(4, n, p=[0.4, 0.3, 0.1, 0.2])   # each station's owner, which this guide leaves out
    rng.integers(1950, 2025, n)                 # and the year it was installed
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    features = np.column_stack([latitude.round(2), longitude.round(2), elevation.round(0), coast_km.round(1)])
    return features.astype("float32"), jan_mean.round(1).astype("float32"), (may_mean < 2.5).astype("float32")


FEATURES = ["latitude", "longitude", "elevation_m", "coast_km"]
features, jan_mean, snow_on_may_1 = station_network_arrays()
order = np.random.default_rng(0).permutation(len(features))
train_rows, valid_rows, test_rows = order[:800], order[800:1000], order[1000:]
mean, std = features[train_rows].mean(axis=0), features[train_rows].std(axis=0)
scaled = (features - mean) / std

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "|", len(features), "stations")

torch 2.11.0 | tensorflow 2.21.0 | 1200 stations


## Worked examples

### Reshaping

The twelve numbers 0 to 11, grouped three ways.

In [2]:
counts = torch.arange(12)
print(counts.reshape(3, 4))
print(counts.reshape(2, -1).shape, counts.view(4, 3).shape)

print(tf.reshape(tf.range(12), (3, 4)))
print(tf.reshape(tf.range(12), (2, -1)).shape)

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
torch.Size([2, 6]) torch.Size([4, 3])
tf.Tensor(
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]], shape=(3, 4), dtype=int32)
(2, 6)


Each shape holds the same twelve values in the same order, grouped differently, and `-1` asks for
whatever size makes the total come out right, 6 here. `view` gives the same result as `reshape` when
the values are already in order, as they are in a tensor just made, and TensorFlow has only
`tf.reshape`.

### A single station as a batch

`table` and `table_tf` hold the stations' features in each framework, and the rest of the notebook
works on them.

In [3]:
table = torch.from_numpy(features)
table_tf = tf.constant(features)

station = table[0]
print(station.shape, station.unsqueeze(0).shape, station[None].shape, table[0:1].shape)
print(table_tf[0].shape, tf.expand_dims(table_tf[0], 0).shape)

torch.Size([4]) torch.Size([1, 4]) torch.Size([1, 4]) torch.Size([1, 4])
(4,) (1, 4)


One station's row has the shape (4,). `unsqueeze(0)`, indexing with `None` and the slice `0:1` each
give it a first dimension of size 1, the shape (1, 4) that a layer expects for a batch of one
station. `tf.expand_dims` does the same in TensorFlow.

In [4]:
print(torch.zeros(1, 4, 1).squeeze().shape, torch.zeros(1, 4, 1).squeeze(0).shape)
print(tf.squeeze(tf.zeros((1, 4, 1))).shape, tf.squeeze(tf.zeros((1, 4, 1)), axis=0).shape)

torch.Size([4]) torch.Size([4, 1])
(4,) (4, 1)


`squeeze` with no argument removes every dimension of size 1, and with a dimension named it removes
only that one. Naming it is safer, since a batch that happens to hold one example has a dimension of
size 1 too.

### Swapping dimensions

In [5]:
print(table.T.shape, table.permute(1, 0).shape, tf.transpose(table_tf).shape)

pictures = torch.zeros(2, 12, 12, 1)
print(pictures.permute(0, 3, 1, 2).shape)
print(tf.transpose(tf.zeros((2, 12, 12, 1)), perm=[0, 3, 1, 2]).shape)
print(table.is_contiguous(), table.T.is_contiguous())

torch.Size([4, 1200]) torch.Size([4, 1200]) (4, 1200)
torch.Size([2, 1, 12, 12])
(2, 1, 12, 12)
True False


`.T` and `permute(1, 0)` swap the table's two dimensions, and `tf.transpose` does the same. A batch
of pictures stored as (batch, height, width, channels), the order TensorFlow's layers use, becomes
(batch, channels, height, width), the order PyTorch's use, with `permute(0, 3, 1, 2)` or
`perm=[0, 3, 1, 2]`, which the **Convolutional Networks** notebook needs. `is_contiguous()` says
whether a tensor's values sit in memory in the order its shape reads them. A transposed tensor's
values do not, which matters to `view` in the Common errors.

### Reducing along a dimension

In [6]:
print(table.mean(dim=0).numpy().round(2))
print(tf.reduce_mean(table_tf, axis=0).numpy().round(2))
print(table.mean(dim=1).shape, table.mean(dim=1, keepdim=True).shape)
print(tf.reduce_mean(table_tf, axis=1).shape, tf.reduce_mean(table_tf, axis=1, keepdims=True).shape)

[ 64.45  17.28 292.8   24.63]
[ 64.45  17.28 292.8   24.63]
torch.Size([1200]) torch.Size([1200, 1])
(1200,) (1200, 1)


The mean over dimension 0 is one number per feature, so the average station sits at latitude 64.45,
longitude 17.28, 292.8 meters up and 24.63 kilometers from the coast. The mean over dimension 1 is one number
per station, 1,200 of them, and `keepdim=True` or `keepdims=True` keeps the second dimension at size
1. TensorFlow's reductions have names that start with `reduce_`.

In [7]:
largest = table.max(dim=0)
print(largest.values)
print(largest.indices)
print(tf.reduce_max(table_tf, axis=0).numpy(), tf.argmax(table_tf, axis=0).numpy())

tensor([  70.9800,   29.4300, 1224.0000,  157.1000])
tensor([1009,  233,   13,  562])
[  70.98   29.43 1224.    157.1 ] [1009  233   13  562]


PyTorch's `max` over a dimension returns two tensors, the largest values and the rows they are in,
so the northernmost station is in row 1009 and the highest, 1,224 meters up, in row 13. TensorFlow
keeps the two apart, in `tf.reduce_max` for the values and `tf.argmax` for the rows.

### Broadcasting

Each line below adds two tensors whose shapes differ, and prints the shape of the sum.

In [8]:
print((torch.ones(3, 1) + torch.ones(1, 4)).shape)
print((torch.ones(1200, 4) + torch.ones(4)).shape)
print((tf.ones((3, 1)) + tf.ones((1, 4))).shape)

torch.Size([3, 4])
torch.Size([1200, 4])
(3, 4)


Compared from the right, (3, 1) and (1, 4) have 1 against 4 and 3 against 1, so each dimension of
size 1 stretches and the sum is (3, 4). (1200, 4) and (4,) match in their last dimension, and the
first dimension that the second lacks counts as 1, so the row of 4 is added to every one of the 1,200
rows.

In [9]:
centered = table - table.mean(dim=0)
standard = centered / table.std(dim=0, correction=0)
print(table.shape, table.mean(dim=0).shape, standard.shape)
print(bool(standard.mean(dim=0).abs().max() < 1e-5), standard.std(dim=0, correction=0).numpy().round(3))

standard_tf = (table_tf - tf.reduce_mean(table_tf, axis=0)) / tf.math.reduce_std(table_tf, axis=0)
print(standard_tf.shape, tf.math.reduce_std(standard_tf, axis=0).numpy().round(3))

torch.Size([1200, 4]) torch.Size([4]) torch.Size([1200, 4])
True [1. 1. 1. 1.]
(1200, 4) [1. 1. 1. 1.]


Subtracting the row of four means centers every feature on 0, and dividing by the row of four
standard deviations gives each a standard deviation of 1. The largest column mean left is below
0.00001, which is the rounding of 32-bit floats. The TensorFlow version gives the same shape and the
same standard deviations.

In [10]:
print("PyTorch, default:   ", table.std(dim=0).numpy().round(4))
print("PyTorch, correction=0:", table.std(dim=0, correction=0).numpy().round(4))
print("NumPy:              ", features.std(axis=0).round(4))
print("TensorFlow:         ", tf.math.reduce_std(table_tf, axis=0).numpy().round(4))

PyTorch, default:    [  3.7248   5.7229 191.1426  24.0811]
PyTorch, correction=0: [  3.7233   5.7205 191.0629  24.0711]
NumPy:               [  3.7233   5.7205 191.0629  24.0711]
TensorFlow:          [  3.7233   5.7205 191.0629  24.0711]


PyTorch's `std` divides by n − 1 by default, which estimates the spread of a population from a
sample, while NumPy and TensorFlow divide by n. With 1,200 stations the two differ in the third
decimal place for latitude, 3.7248 against 3.7233. `correction=0` makes PyTorch divide by n, as
NumPy did when Setup made `scaled`.

### Joining tensors

In [11]:
print(torch.cat([table[:2], table[2:5]]).shape, torch.stack([table[0], table[1]]).shape)
print(tf.concat([table_tf[:2], table_tf[2:5]], axis=0).shape, tf.stack([table_tf[0], table_tf[1]]).shape)

torch.Size([5, 4]) torch.Size([2, 4])
(5, 4) (2, 4)


`cat` and `concat` join tensors along a dimension they already have, so 2 rows and 3 rows make 5.
`stack` adds a new dimension, so two rows of 4 make a tensor of shape (2, 4).

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/03-shapes-and-broadcasting-solutions.ipynb).

**1.** Reshape `torch.arange(24)` into the shape (2, 3, 4), then into (4, -1), and print both shapes. Do the same with `tf.range` and `tf.reshape`.

In [12]:
# your code here

**2.** Take the station in row 10 of `features` as a batch of one, in PyTorch and in TensorFlow, and print both shapes.

In [13]:
# your code here

**3.** Find the lowest latitude among the stations and the row it is in, in PyTorch with `min(dim=0)` and in TensorFlow with `tf.reduce_min` and `tf.argmin`.

In [14]:
# your code here

**4.** Take the elevation column of `table_tf` with a slice, so that it keeps the shape (1200, 1), subtract the mean elevation found with `tf.reduce_mean` and `keepdims=True`, and print the result's shape and its first three values rounded to one place.

In [15]:
# your code here

**5.** Use broadcasting on purpose, subtracting the stations' latitudes as a row of shape (1, 1200) from the same latitudes as a column of shape (1200, 1). Print the result's shape and its largest value, and check that value against the largest latitude minus the smallest.

In [16]:
# your code here

## Common errors

### `RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.`

The features as a table of 4 rows, one per feature, flattened into one long row:

In [17]:
by_feature = table.T
by_feature.view(-1)

RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

`table.T` reads the same memory with its dimensions swapped, so one feature's values are not next to
each other in memory. `view` never copies, and no single way of reading that memory gives the 4,800
values in the new order, so it raises, and the message names the fix.

In [18]:
print(by_feature.reshape(-1).shape, by_feature.contiguous().view(-1).shape)

torch.Size([4800]) torch.Size([4800])


`reshape` copies the values into the new order when it has to, and `contiguous()` makes that copy
explicitly, after which `view` works.

### `RuntimeError: The size of tensor a (4) must match the size of tensor b (1200) at non-singleton dimension 1`

The standardized features in `scaled` put latitude, longitude, elevation and the coast on one
scale, so a station's four values can be compared with one another by subtracting their mean. Each
station's values, minus its own mean:

In [19]:
scaled_table = torch.from_numpy(scaled)
scaled_table - scaled_table.mean(dim=1)

RuntimeError: The size of tensor a (4) must match the size of tensor b (1200) at non-singleton dimension 1

`scaled_table` is (1200, 4), and its mean over dimension 1 is (1200,). Compared from the right, the
last dimensions are 4 and 1200, and neither is 1, so broadcasting fails. Tensor a is the left side of
the subtraction and b the right, and dimension 1 is where they clash. Keeping the reduced dimension
makes the means a column, (1200, 1), which stretches across the four features, and the result has
the table's shape.

In [20]:
print((scaled_table - scaled_table.mean(dim=1, keepdim=True)).shape)

torch.Size([1200, 4])


### `InvalidArgumentError: {{function_node __wrapped__Sub_device_/job:localhost/replica:0/task:0/device:CPU:0}} Incompatible shapes: [1200,4] vs. [1200] [Op:Sub] name:`

The same mistake in TensorFlow:

In [21]:
scaled_tf = tf.constant(scaled)
scaled_tf - tf.reduce_mean(scaled_tf, axis=1)

InvalidArgumentError: {{function_node __wrapped__Sub_device_/job:localhost/replica:0/task:0/device:CPU:0}} Incompatible shapes: [1200,4] vs. [1200] [Op:Sub] name: 

TensorFlow's message starts with the operation that failed and the device it ran on, here `Sub` on
the CPU, and then gives the two shapes. The fix is the same, `keepdims=True`.

In [22]:
print((scaled_tf - tf.reduce_mean(scaled_tf, axis=1, keepdims=True)).shape)

(1200, 4)


### No error, and 1,440,000 differences where 1,200 were meant

A rough rule predicts each station's January mean from its latitude alone. `table[:, 0:1]` takes the
latitude column with a slice, which keeps the dimension, so the predictions have the shape (1200, 1).

In [23]:
predicted = 1.5 - 0.55 * (table[:, 0:1] - 58)
actual = torch.from_numpy(jan_mean)
errors = predicted - actual
print(predicted.shape, actual.shape, errors.shape)
print("mean squared error:", round((errors ** 2).mean().item(), 3))

torch.Size([1200, 1]) torch.Size([1200]) torch.Size([1200, 1200])
mean squared error: 22.398


The predictions are (1200, 1) and the answers (1200,). Compared from the right, the 1 stretches to
1,200 and the dimension the answers lack counts as 1, so every answer is subtracted from every
prediction, and the mean of those 1,440,000 squares comes out as 22.398 with nothing in it to say
that it is wrong.

In [24]:
errors = predicted.squeeze(1) - actual
print(errors.shape, "mean squared error:", round((errors ** 2).mean().item(), 3))

predicted_tf = 1.5 - 0.55 * (table_tf[:, 0:1] - 58)
print((predicted_tf - tf.constant(jan_mean)).shape)
errors_tf = tf.squeeze(predicted_tf, axis=1) - tf.constant(jan_mean)
print(errors_tf.shape, "mean squared error:", round(float(tf.reduce_mean(errors_tf ** 2)), 3))

torch.Size([1200]) mean squared error: 14.067
(1200, 1200)
(1200,) mean squared error: 14.067


TensorFlow broadcasts the column against the vector the same way, to (1200, 1200). Squeezed to
(1200,), each prediction meets its own station's answer, and the mean squared error is 14.067 in
both frameworks. Printing the shapes of both sides before taking a mean catches this,
since the mistake itself raises nothing.

## Recap

- You can reshape a tensor with `reshape` or `view`, and know when `view` needs a copy first.
- You can add and remove a dimension of size 1, to make one example a batch of one.
- You can reduce along a dimension and keep it, so the result broadcasts back.
- You can predict the shape broadcasting gives, and catch a column minus a vector before it trains.

## What is next

The **Automatic Differentiation** notebook shows what PyTorch and TensorFlow record as a value is
computed, and how each one walks that record back to find a gradient.

---

&#8592; **Previous:** [Tensors](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/02-tensors.ipynb)  &nbsp;·&nbsp;  [PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Automatic Differentiation](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/04-automatic-differentiation.ipynb) &#8594;
